In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from urllib.request import urlopen
import json
import requests
import time
import glob
import os

#helps us with organizing the data
import duckdb

from sklearn.preprocessing import StandardScaler

In [ ]:
f1_data_cleaned = pd.read_csv("/content/lap_features_labeled.csv")
f1_data_cleaned.head()

,session_key,driver_number,lap_number,date_start,lap_duration,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,...,speed_std,pct_above_300,pct_200_300,pct_100_200,pct_below_100,pct_drs_active,n_samples,n_gear_changes,team_name,driver_name
0,7768,16,5,2023-03-04 15:15:09.670,91.094,29.099,39.134,22.861,242.0,274.0,...,70.944592,0.106017,0.498567,0.306590,0.088825,0.234957,349,45.0,Ferrari,Charles LECLERC
1,7768,16,10,2023-03-04 15:38:30.536,91.699,29.301,39.434,22.964,241.0,272.0,...,71.882821,0.088068,0.511364,0.303977,0.096591,0.235795,352,49.0,Ferrari,Charles LECLERC
2,7768,16,16,2023-03-04 15:59:35.856,90.000,28.825,38.614,22.561,243.0,273.0,...,71.236785,0.105263,0.526316,0.274854,0.093567,0.219298,342,47.0,Ferrari,Charles LECLERC
3,7768,18,2,2023-03-04 15:14:18.031,91.617,29.266,39.428,22.923,241.0,268.0,...,68.925811,0.066474,0.534682,0.309249,0.089595,0.242775,346,44.0,Aston Martin,Lance STROLL
4,7768,18,8,2023-03-04 15:37:31.240,92.305,29.581,39.779,22.945,241.0,267.0,...,67.705949,0.066482,0.526316,0.321330,0.085873,0.238227,361,47.0,Aston Martin,Lance STROLL


In [ ]:
print(f1_data_cleaned.shape)

df = f1_data_cleaned.dropna().reset_index(drop=True)
print(f1_data_cleaned.shape)

df.head()

(1506, 34)
(1506, 34)


,session_key,driver_number,lap_number,date_start,lap_duration,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,...,speed_std,pct_above_300,pct_200_300,pct_100_200,pct_below_100,pct_drs_active,n_samples,n_gear_changes,team_name,driver_name
0,7768,16,5,2023-03-04 15:15:09.670,91.094,29.099,39.134,22.861,242.0,274.0,...,70.944592,0.106017,0.498567,0.306590,0.088825,0.234957,349,45.0,Ferrari,Charles LECLERC
1,7768,16,10,2023-03-04 15:38:30.536,91.699,29.301,39.434,22.964,241.0,272.0,...,71.882821,0.088068,0.511364,0.303977,0.096591,0.235795,352,49.0,Ferrari,Charles LECLERC
2,7768,16,16,2023-03-04 15:59:35.856,90.000,28.825,38.614,22.561,243.0,273.0,...,71.236785,0.105263,0.526316,0.274854,0.093567,0.219298,342,47.0,Ferrari,Charles LECLERC
3,7768,18,2,2023-03-04 15:14:18.031,91.617,29.266,39.428,22.923,241.0,268.0,...,68.925811,0.066474,0.534682,0.309249,0.089595,0.242775,346,44.0,Aston Martin,Lance STROLL
4,7768,18,8,2023-03-04 15:37:31.240,92.305,29.581,39.779,22.945,241.0,267.0,...,67.705949,0.066482,0.526316,0.321330,0.085873,0.238227,361,47.0,Aston Martin,Lance STROLL


In [ ]:
label_cols = [
    "team_name", "driver_name", "driver_number",
    "session_key", "lap_number", "date_start"   # add date_start
]
feature_cols = [c for c in f1_data_cleaned.columns
                if c not in label_cols and c != "n_samples"]

X = df[feature_cols].values
y_team = df["team_name"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
teams = sorted(set(y_team)) #sorts the teams and removes any duplicates
team_to_id = {name: i for i, name in enumerate(teams)}
y_team_id = np.array([team_to_id[t] for t in y_team])

print("Mapping Teams:")
for name, i in team_to_id.items():
  print(f"{i}: {name}")

combined = np.column_stack([X_scaled, y_team_id])

restrictions = ["1"] * len(feature_cols) + ["0"]

# saved into a csv file for easy usage for future use
OUT_PATH = "f1_laps_scaled.csv"
with open(OUT_PATH, "w") as f:
    f.write(",".join(restrictions) + "\n")
    np.savetxt(f, combined, delimiter=",", fmt="%.6f")

print(f"\nSaved {len(combined)} rows × {combined.shape[1]} cols to {OUT_PATH}")

Mapping Teams:
0: Alfa Romeo
1: AlphaTauri
2: Alpine
3: Aston Martin
4: Ferrari
5: Haas F1 Team
6: McLaren
7: Mercedes
8: Red Bull Racing
9: Williams

Saved 1502 rows × 28 cols to f1_laps_scaled.csv


In [ ]:
%%writefile clusterHelper.py

import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score, silhouette_samples, rand_score

def loadDataset(f):
    # read first line manually to get the number of real columns
    with open(f, "r") as file:
        firstLine = file.readline().strip()

    restrictions = np.array([int(x.strip()) for x in firstLine.split(",") if x.strip() != ""])
    numCols = len(restrictions)

    # usecols ignores accidental trailing empty columns, like the extra comma in AccidentsSet03.csv
    df = pd.read_csv(f, header=None, usecols=range(numCols), skipinitialspace=True)

    data = df.iloc[1:].reset_index(drop=True)

    colsUsed = [i for i, val in enumerate(restrictions) if val == 1]
    colsIgnored = [i for i, val in enumerate(restrictions) if val == 0]
    X = data.iloc[:, colsUsed].astype(float).to_numpy()

    ignored = None
    if len(colsIgnored) > 0:
        ignored = data.iloc[:, colsIgnored].to_numpy()

    return X, ignored, restrictions

def euclidean(a, b):
  return np.sqrt(np.sum((a - b) ** 2))

def centroid(pts):
  return np.mean(pts, axis = 0)

def clusterRad(pts, centroid):
  # returns max dist from centroid to any cluster point
  if len(pts) == 0:
    return 0
  return max([euclidean(pt, centroid) for pt in pts])

def interclusterDist(c1, c2):
  return euclidean(c1, c2)

def radInterclusterRatio(X, labels):
  # average cluster radius / minimum distance between cluster centroids
  # want this number to be small
  unique = [label for label in sorted(set(labels)) if label != -1]

  if len(unique) < 2:
    return None

  centroids = {}
  radii = {}

  for label in unique:
    pts = X[np.array(labels) == label]
    c = centroid(pts)
    rad = clusterRad(pts, c)
    centroids[label] = c
    radii[label] = rad

  avgRad = np.mean(list(radii.values()))

  minInterclusterDist = float("inf")
  for i in range(len(unique)):
    for j in range(i + 1, len(unique)):
      dist = interclusterDist(centroids[unique[i]], centroids[unique[j]])
      if dist < minInterclusterDist:
        minInterclusterDist = dist

  if minInterclusterDist == 0:
    return None

  return avgRad / minInterclusterDist

def printReport(X, labels, groundTruth = None):
  # prints general metrics and cluster-by-cluster metrics
  labels = np.array(labels)
  unique = sorted(set(labels))

  print("--Overall Cluster Report")
  print(f"# of data points: {len(X)}")
  print(f"Cluster labels found: {[int(label) for label in unique]}")

  nonNoise = [label for label in unique if label != -1]
  if len(nonNoise) >= 2:
    ratio = radInterclusterRatio(X, labels)
    print(f"Radius / Intercluster Distance Ratio: {ratio}")

    try:
      silhouetteScore = silhouette_score(X, labels)
      print(f"Overall Silhouette Score: {silhouetteScore}")
    except Exception as e:
      print(f"Silhouette score could not be computed ({e})")
  else:
    print("Not enough clusters for silhouette score")
    print("Not enough clusters for radius / intercluster distance ratio")

  if groundTruth is not None:
    try:
      groundTruth = groundTruth.flatten()
      randIdx = rand_score(groundTruth, labels)
      print(f"Rand Index: {randIdx}")
    except Exception as e:
      print(f"Rand index could not be computed ({e})")

  print("\n--Cluster Details--")

  try:
    silSamples = silhouette_samples(X, labels)
  except Exception as e:
    silSamples = None

  for label in unique:
    pts = X[labels == label]

    if label == -1:
      print("\n--Outliers / Noise--")
    else:
      print(f"\n--Cluster {label}--")

    print(f"# of pts: {len(pts)}")
    if len(pts) > 0 and label != -1:
      c = centroid(pts)
      rad = clusterRad(pts, c)
      print(f"Centroid: {c}")
      print(f"Radius: {rad}")

      if silSamples is not None:
        silClusters = silSamples[labels == label]
        print(f"Cluster Silhouette Score: {np.mean(silClusters)}")

    if len(pts) <= 10:
      print("Points: ")
      for pt in pts:
        print(pt)
    else:
      print("Representative points: ")
      for pt in pts[:3]:
        print(pt)


Writing clusterHelper.py


In [ ]:
%%writefile kmeans.py

import sys
import numpy as np
from clusterHelper import loadDataset, euclidean, printReport

def initializeCentroids(X, k):
  # randomly choose k points and set them as the centroids
  idxs = np.random.choice(len(X), size = k, replace = False)
  centroids = X[idxs]
  return centroids

def ptsToClusters(X, centroids):
  # find nearest centroid for each point
  labels = []
  for pt in X:
    dist = [euclidean(pt, centroid) for centroid in centroids]
    labels.append(np.argmin(dist))
  return np.array(labels)

def recompCentroids(X, labels, k, prevCentroids):
  # recompute each centroid as the mean of all cluster points
  newCentroids = []
  for id in range(k):
    pts = X[labels == id]
    if len(pts) == 0:
      newCentroids.append(prevCentroids[id])
    else:
      newCentroids.append(np.mean(pts, axis = 0))
  return np.array(newCentroids)

def kmeans(X, k, maxIter = 100):
  centroids = initializeCentroids(X, k)
  for iter in range(maxIter):
    labels = ptsToClusters(X, centroids)
    newCentroids = recompCentroids(X, labels, k, centroids)
    if np.allclose(centroids, newCentroids):
      print(f"k-means converged after {iter + 1} iterations")
      break
    centroids = newCentroids
  return labels, centroids

def main():
  if len(sys.argv) < 3:
    print("python3 kmeans.py <dataset> <k>")
    return

  f = sys.argv[1]
  k = int(sys.argv[2])

  X, ignored, restrictions = loadDataset(f)

  labels, centroids = kmeans(X, k)

  groundTruth = None
  if ignored is not None and ignored.shape[1] == 1:
      groundTruth = ignored

  printReport(X, labels, groundTruth)

if __name__ == "__main__":
  main()

Writing kmeans.py


In [ ]:
%%writefile hclustering.py

import sys
from clusterHelper import loadDataset, euclidean, printReport
import numpy as np
import json


def compute_distance_matrix(X):
  """
  Returns an (n x n) symmetric matrix of pairwise Euclidean distances.
  """
  n = len(X)
  D = np.zeros((n, n))

  for i in range(n):
    for j in range(i + 1, n):
      d = euclidean(X[i], X[j])
      D[i, j] = d
      D[j, i] = d

  return D


def cluster_distance(a_indices, b_indices, D, method="single"):
  """
  Computes the distance between two clusters using the distance matrix D.
  """
  distances = [D[i, j] for i in a_indices for j in b_indices]

  if method == "single":
    return min(distances)
  elif method == "complete":
    return max(distances)
  elif method == "average":
    return sum(distances) / len(distances)
  else:
    raise ValueError(f"Unknown linkage method: '{method}'")


def make_leaf(point, index):
  """
  Create a leaf node.
  The index is needed so we know which original data point gets assigned later.
  """
  return {
      "type": "leaf",
      "index": int(index),
      "data": list(point)
  }


def makeNode(height, left, right):
  return {
      "type": "node",
      "height": round(float(height), 6),
      "nodes": [left, right]
  }


def hclustering(X, D, linkage="single"):
  n = len(X)

  nodes = {i: make_leaf(X[i], i) for i in range(n)}
  indices = {i: [i] for i in range(n)}
  active = list(range(n))
  nextId = n

  while len(active) > 1:
    bestDist = float("inf")
    bestA, bestB = None, None

    for i in range(len(active)):
      for j in range(i + 1, len(active)):
        ca, cb = active[i], active[j]
        d = cluster_distance(indices[ca], indices[cb], D, linkage)

        if d < bestDist:
          bestDist = d
          bestA, bestB = ca, cb

    newNode = makeNode(bestDist, nodes[bestA], nodes[bestB])
    newIndices = indices[bestA] + indices[bestB]

    nodes[nextId] = newNode
    indices[nextId] = newIndices

    active.remove(bestA)
    active.remove(bestB)
    active.append(nextId)

    nextId += 1

  root = nodes[active[0]]
  root["type"] = "root"

  return root


def collectLeaves(node):
  """
  Collects the original indexes of all leaf nodes under this node.
  """
  if node["type"] == "leaf":
    return [node["index"]]

  leaves = []

  for child in node["nodes"]:
    leaves.extend(collectLeaves(child))

  return leaves


def cutTree(node, threshold, labels, clusterID):
  """
  Cuts the dendrogram at the given threshold and assigns labels.

  If a node's height is <= threshold, everything under that node becomes
  one cluster.
  """
  if node["type"] == "leaf":
    labels[node["index"]] = clusterID
    return clusterID + 1

  if node["height"] <= threshold:
    leafIndexes = collectLeaves(node)

    for idx in leafIndexes:
      labels[idx] = clusterID

    return clusterID + 1

  for child in node["nodes"]:
    clusterID = cutTree(child, threshold, labels, clusterID)

  return clusterID


def labelsToClusters(X, labels):
  """
  Converts labels array into list of clusters, where each cluster is a list of points.
  """
  clusters = []

  for label in sorted(set(labels)):
    pts = X[labels == label]
    clusters.append([list(p) for p in pts])

  return clusters


# PRINT HELPERS
def printDendrogram(root):
  dendroJson = json.dumps(root, indent=2)

  print("\nDendrogram")
  print(dendroJson)

  with open("dendrogram.json", "w") as f:
    f.write(dendroJson)

  print("\nDendrogram saved to dendrogram.json")


def printClusters(clusters, threshold, n):
  print(f"\n--Clusters at threshold {threshold}--")
  print(f"# of clusters: {len(clusters)}")
  print(f"# of data points: {n}")

  for i, cluster in enumerate(clusters):
    pts = [np.array(p) for p in cluster]
    centroid = np.mean(pts, axis=0).tolist()
    pct = len(cluster) / n * 100
    cStr = ", ".join(f"{v:.4f}" for v in centroid)

    print(f"\nCluster {i}: {len(cluster)} pts ({pct:.2f}%) centroid=[{cStr}]")

    MAX_SHOW = 19

    for pt in cluster[:MAX_SHOW]:
      print(" ", [round(float(v), 4) for v in pt])

    if len(cluster) > MAX_SHOW:
      print(f"    ... and {len(cluster) - MAX_SHOW} more")


def main():
  if len(sys.argv) < 2:
    print("Usage: python3 hclustering.py <dataset> [threshold] [--linkage single|complete|average]")
    return

  f = sys.argv[1]
  threshold = None
  linkage = "single"

  i = 2

  while i < len(sys.argv):
    if sys.argv[i] == "--linkage" and i + 1 < len(sys.argv):
      linkage = sys.argv[i + 1]
      i += 2
    else:
      try:
        threshold = float(sys.argv[i])
      except ValueError:
        print(f"Ignoring argument: {sys.argv[i]}")
      i += 1

  X, ignored, restrictions = loadDataset(f)

  print(f"Loaded {len(X)} points from '{f}'")
  print(f"Linkage: {linkage}")

  D = compute_distance_matrix(X)
  root = hclustering(X, D, linkage)

  printDendrogram(root)

  if threshold is not None:
    labels = np.full(len(X), -1)

    cutTree(root, threshold, labels, 0)

    print("\nUnassigned points:", np.sum(labels == -1))

    clusters = labelsToClusters(X, labels)

    groundTruth = None
    if ignored is not None and ignored.shape[1] == 1:
      groundTruth = ignored

    printClusters(clusters, threshold, len(X))
    printReport(X, labels, groundTruth)


if __name__ == "__main__":
  main()

Writing hclustering.py


In [ ]:
%%writefile dbscan.py

import sys
import numpy as np
from clusterHelper import loadDataset, printReport, euclidean

def getNeighbors(X, ptIdx, e):
    # returns a list of indexes of all points within e distance of ptIdx

    neighborsList = []

    for i in range(len(X)):
        dist = euclidean(X[ptIdx], X[i])
        if dist <= e:
            neighborsList.append(i)

    return neighborsList

def dbscan(X, e, minPts):
    # DBSCAN clustering algorithm.
    # -1 -> noise/outlier
    # 0, 1, 2, ... -> cluster labels

    labels = np.full(len(X), -1)
    visited = np.full(len(X), False)

    id = 0

    for ptIdx in range(len(X)):
        if visited[ptIdx]:
            continue
        visited[ptIdx] = True

        neighborsList = getNeighbors(X, ptIdx, e)

        if len(neighborsList) < minPts:
            labels[ptIdx] = -1
        else:
            expandCluster(X, labels, visited, ptIdx, neighborsList, id, e, minPts)
            id += 1

    return labels


def expandCluster(X, labels, visited, ptIdx, neighborsList, id, e, minPts):
    # from core point, expand cluster

    labels[ptIdx] = id

    i = 0
    while i < len(neighborsList):
        neighborIdx = neighborsList[i]

        if not visited[neighborIdx]:
            visited[neighborIdx] = True
            nNeighbors = getNeighbors(X, neighborIdx, e)

            if len(nNeighbors) >= minPts:
                for n in nNeighbors:
                    if n not in neighborsList:
                        neighborsList.append(n)
        if labels[neighborIdx] == -1:
            labels[neighborIdx] = id

        i += 1

def printOutlierSummary(labels):
    labels = np.array(labels)
    totalPts = len(labels)
    numOutliers = np.sum(labels == -1)
    percentOutliers = (numOutliers / totalPts) * 100
    print("--DBSCAN Outlier Summary--")
    print(f"# of outliers: {numOutliers}")
    print(f"Percent outliers: {percentOutliers:.2f}%")

def main():
    if len(sys.argv) < 4:
        print("python3 dbscan.py <f> <e> <minPts>")
        return

    f = sys.argv[1]
    e = float(sys.argv[2])
    minPts = int(sys.argv[3])

    X, ignored, restrictions = loadDataset(f)

    labels = dbscan(X, e, minPts)

    groundTruth = None
    if ignored is not None and ignored.shape[1] == 1:
      groundTruth = ignored

    print(f"DBSCAN finished with epsilon = {e}, minPts = {minPts}")
    printOutlierSummary(labels)
    printReport(X, labels, groundTruth)


if __name__ == "__main__":
    main()

Writing dbscan.py


In [ ]:
%%writefile outputs.py

import os
import re
import pandas as pd

def getMetrics(pattern, text):
  match = re.search(pattern, text)
  if match:
    return match.group(1)
  return None


def fSummary(fPath):
  fileName = os.path.basename(fPath)
  with open(fPath, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

  silhouette = getMetrics(r"Overall Silhouette Score: \s*([-+]?\d*\.\d+|\d+)", text)
  rand = getMetrics(r"Rand Index: \s*([-+]?\d*\.\d+|\d+)", text)
  ratio = getMetrics(r"Radius / Intercluster Distance Ratio: \s*([-+]?\d*\.\d+|\d+)", text)
  dataPts = getMetrics(r"# of data points: \s*(\d+)", text)
  outliers = getMetrics(r"# of outliers: \s*(\d+)", text)
  clusterLabels = getMetrics(r"Cluster labels found: \s*(\[.*?\])", text)

  return {"file": fileName,
          "silhouette": float(silhouette) if silhouette else None,
          "rand_index": float(rand) if rand else None,
          "radius_intercluster_ratio": float(ratio) if ratio else None,
          "dataPts": int(dataPts) if dataPts else None,
          "outliers": int(outliers) if outliers else None,
          "clusterLabels": clusterLabels}


def main():
  rows = []

  for fileName in os.listdir("outputs"):
    if fileName.endswith(".txt"):
      fPath = os.path.join("outputs", fileName)
      rows.append(fSummary(fPath))

  df = pd.DataFrame(rows)
  df = df.sort_values(by="file")
  df.to_csv("results.csv", index=False)
  print(df)
  print("\nSaved to results.csv")

if __name__ == "__main__":
  main()

Writing outputs.py


In [ ]:
%%writefile visualize_clusters.py

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from clusterHelper import loadDataset
from kmeans import kmeans

bestKmeans = {
  "iris.csv": 3,
  "4clusters.csv": 4,
  "mammal_milk.csv": 2,
  "planets.csv": 2,
  "AccidentsSet03.csv": 3}

def makePlot(filename):
  X, ignored, restrictions = loadDataset(filename)
  basename = os.path.basename(filename)

  if basename not in bestKmeans:
    print(f"No k value saved for {basename}")
    return

  k = bestKmeans[basename]
  labels, centroids = kmeans(X, k)
  plt.figure(figsize=(7, 5))

  # first two clustering columns for visualization
  plt.scatter(X[:, 0], X[:, 1], c=labels)
  # plot centroids if they have 2+ dimensions
  if centroids.shape[1] >= 2:
    plt.scatter(centroids[:, 0], centroids[:, 1], marker="x", s=120, linewidths=3)

  plt.title(f"K-means Clustering: {basename}, k={k}")
  plt.xlabel("Feature 1")
  plt.ylabel("Feature 2")
  os.makedirs("figures", exist_ok=True)
  outname = basename.replace(".csv", "_kmeans.png")
  outpath = os.path.join("figures", outname)
  plt.savefig(outpath, bbox_inches="tight", dpi=200)
  plt.close()
  print(f"Saved {outpath}")

def main():
  if len(sys.argv) >= 2:
    makePlot(sys.argv[1])
  else:
    datasets = [
      "iris.csv",
      "4clusters.csv",
      "mammal_milk.csv",
      "planets.csv",
      "AccidentsSet03.csv"]

    for dataset in datasets:
      if os.path.exists(dataset):
        makePlot(dataset)
      else:
        print(f"Skipping {dataset}: file not found.")

if __name__ == "__main__":
  main()

Writing visualize_clusters.py


In [ ]:
%%writefile visualize_clusters.py

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from clusterHelper import loadDataset
from kmeans import kmeans

FILENAME = "f1_laps_scaled.csv"
K = 10  # one cluster per team

# Team ID → name (must match the order from your earlier mapping)
TEAM_NAMES = [
  "Alfa Romeo",
  "Alpine",
  "AlphaTauri",
  "Aston Martin",
  "Ferrari",
  "Haas F1 Team",
  "McLaren",
  "Mercedes",
  "Red Bull Racing",
  "Williams"
]

def makePlot(filename, k):
  X, ignored, restrictions = loadDataset(filename)
  basename = os.path.basename(filename)

  # Run k-means in the FULL feature space (all 22 dims)
  labels, centroids = kmeans(X, k)

  # Project to 2D with PCA for visualization
  pca = PCA(n_components=2)
  X_2d = pca.fit_transform(X)
  centroids_2d = pca.transform(centroids)

  variance_explained = pca.explained_variance_ratio_.sum() * 100

  os.makedirs("figures", exist_ok=True)

  # ---- Plot 1: colored by k-means cluster assignment ----
  plt.figure(figsize=(9, 6))
  plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap="tab10", s=15, alpha=0.7)
  plt.scatter(centroids_2d[:, 0], centroids_2d[:, 1],
              marker="x", s=200, linewidths=3, c="black", label="Centroids")
  plt.title(f"K-means Clustering on F1 Telemetry (k={k})\n"
            f"PCA projection — {variance_explained:.1f}% variance explained")
  plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
  plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
  plt.legend()
  out1 = os.path.join("figures", "f1_kmeans_clusters.png")
  plt.savefig(out1, bbox_inches="tight", dpi=200)
  plt.close()
  print(f"Saved {out1}")

  # ---- Plot 2: colored by TRUE team label (for comparison) ----
  if ignored is not None and ignored.shape[1] == 1:
    team_ids = ignored.flatten().astype(int)

    plt.figure(figsize=(9, 6))
    for team_id in sorted(set(team_ids)):
      mask = team_ids == team_id
      label = TEAM_NAMES[team_id] if team_id < len(TEAM_NAMES) else f"Team {team_id}"
      plt.scatter(X_2d[mask, 0], X_2d[mask, 1], s=15, alpha=0.7, label=label)
    plt.title("F1 Telemetry Colored by True Team Label\n"
              f"PCA projection — {variance_explained:.1f}% variance explained")
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    out2 = os.path.join("figures", "f1_true_teams.png")
    plt.savefig(out2, bbox_inches="tight", dpi=200)
    plt.close()
    print(f"Saved {out2}")

def main():
  filename = sys.argv[1] if len(sys.argv) >= 2 else FILENAME
  k = int(sys.argv[2]) if len(sys.argv) >= 3 else K

  if not os.path.exists(filename):
    print(f"File not found: {filename}")
    return

  makePlot(filename, k)

if __name__ == "__main__":
  main()

Overwriting visualize_clusters.py


In [ ]:
os.makedirs("outputs", exist_ok=True)

# K-means at k = 10 (one cluster per team)
!python3 kmeans.py f1_laps_scaled.csv 10 > outputs/kmeans_k10.txt

# DBSCAN — start moderate
!python3 dbscan.py f1_laps_scaled.csv 2.0 10 > outputs/dbscan_e2_mp10.txt

# Hierarchical
!python3 hclustering.py f1_laps_scaled.csv 5.0 --linkage average > outputs/hclust_avg_t5.txt

In [ ]:
!python3 outputs.py

results = pd.read_csv("results.csv")
results = results.sort_values("rand_index", ascending=False)
results  #check the cluster labels because -1 is a possible noise in the data

                 file  ...                                      clusterLabels
2  dbscan_e2_mp10.txt  ...  [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,...
1   hclust_avg_t5.txt  ...                 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
0      kmeans_k10.txt  ...                     [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

[3 rows x 7 columns]

Saved to results.csv


,file,silhouette,rand_index,radius_intercluster_ratio,dataPts,outliers,clusterLabels
0,dbscan_e2_mp10.txt,0.478532,0.847784,1.117310,1502,42.0,"[-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,..."
2,kmeans_k10.txt,0.358620,0.805019,2.119600,1502,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]"
1,hclust_avg_t5.txt,0.432316,0.775272,0.721715,1502,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"


In [ ]:
!python3 visualize_clusters.py

k-means converged after 5 iterations
Saved figures/f1_kmeans_clusters.png
Saved figures/f1_true_teams.png


In [ ]:
!ls -lh
!ls -lh outputs
!ls -lh figures
!head results.csv

total 5.3M
-rw-r--r-- 1 root root 4.0K Jun  7 23:03 clusterHelper.py
-rw-r--r-- 1 root root 2.4K Jun  7 23:05 dbscan.py
-rw-r--r-- 1 root root 4.3M Jun  7 23:24 dendrogram.json
-rw-r--r-- 1 root root 390K Jun  7 23:02 f1_laps_scaled.csv
drwxr-xr-x 2 root root 4.0K Jun  7 23:25 figures
-rw-r--r-- 1 root root 5.2K Jun  7 23:04 hclustering.py
-rw-r--r-- 1 root root 1.6K Jun  7 23:03 kmeans.py
-rw-r--r-- 1 root root 600K Jun  7 23:00 lap_features_labeled.csv
drwxr-xr-x 2 root root 4.0K Jun  7 23:10 outputs
-rw-r--r-- 1 root root 1.5K Jun  7 23:05 outputs.py
drwxr-xr-x 2 root root 4.0K Jun  7 23:25 __pycache__
-rw-r--r-- 1 root root  461 Jun  7 23:25 results.csv
drwxr-xr-x 1 root root 4.0K Jun  4 13:32 sample_data
-rw-r--r-- 1 root root 2.9K Jun  7 23:08 visualize_clusters.py
total 4.4M
-rw-r--r-- 1 root root  22K Jun  7 23:10 dbscan_e2_mp10.txt
-rw-r--r-- 1 root root 4.3M Jun  7 23:24 hclust_avg_t5.txt
-rw-r--r-- 1 root root  13K Jun  7 23:10 kmeans_k10.txt
total 336K
-rw-r--r-- 1 root roo